In [8]:
import os
import numpy as np
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, precision_score, recall_score
)

np.random.seed(42)

PROCESSED_DIR = r"C:\Project_EquiGuard\data\processed"
MODELS_DIR    = r"C:\Project_EquiGuard\outputs\models"
ARRAYS_DIR    = os.path.join(PROCESSED_DIR, "arrays")
FIGURES_DIR   = r"C:\Project_EquiGuard\outputs\figures"
LOGS_DIR      = r"C:\Project_EquiGuard\logs"

print("Libraries loaded.")

Libraries loaded.


In [9]:
# Load model, threshold, test data

final_model     = joblib.load(os.path.join(MODELS_DIR, "final_model.pkl"))
best_threshold  = joblib.load(os.path.join(MODELS_DIR, "best_threshold.pkl"))
feature_names   = joblib.load(os.path.join(MODELS_DIR, "feature_names_final.pkl"))

X_test = np.load(os.path.join(ARRAYS_DIR, "X_test_scaled.npy"))
y_test = np.load(os.path.join(ARRAYS_DIR, "y_test.npy"))

print(f"X_test: {X_test.shape}")
print(f"y_test: {np.unique(y_test, return_counts=True)}")
print(f"Threshold: {best_threshold}")

X_test: (133, 29)
y_test: (array([0, 1]), array([ 10, 123]))
Threshold: 0.3


In [10]:
# Final predictions on test set

y_prob_test = final_model.predict_proba(X_test)[:, 1]
y_pred_test = (y_prob_test >= best_threshold).astype(int)

print("=== FINAL TEST SET RESULTS ===")
print(classification_report(y_test, y_pred_test,
      target_names=["Sound", "Lame"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_test):.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_test))

=== FINAL TEST SET RESULTS ===
              precision    recall  f1-score   support

       Sound       0.50      0.10      0.17        10
        Lame       0.93      0.99      0.96       123

    accuracy                           0.92       133
   macro avg       0.72      0.55      0.56       133
weighted avg       0.90      0.92      0.90       133

ROC-AUC: 0.8593

Confusion Matrix:
[[  1   9]
 [  1 122]]


In [11]:
# Confusion matrix figure

cm = confusion_matrix(y_test, y_pred_test)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Sound", "Lame"],
            yticklabels=["Sound", "Lame"],
            ax=ax, cbar=False,
            annot_kws={"size": 14})

ax.set_xlabel("Predicted Label", fontsize=12)
ax.set_ylabel("True Label", fontsize=12)
ax.set_title("Confusion Matrix — Test Set", fontsize=13)

plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "confusion_matrix.png")
plt.savefig(fig_path, dpi=300)
plt.close()
print("Confusion matrix saved.")

Confusion matrix saved.


In [12]:
# Feature importance figure

importances = final_model.feature_importances_
top_idx     = np.argsort(importances)[::-1][:15]
top_names   = [feature_names[i] for i in top_idx]
top_vals    = importances[top_idx]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(range(len(top_idx)), top_vals[::-1], color="#1976D2")
ax.set_yticks(range(len(top_idx)))
ax.set_yticklabels(top_names[::-1], fontsize=10)
ax.set_xlabel("Feature Importance", fontsize=12)
ax.set_title("Top 15 Feature Importances — Random Forest", fontsize=13)
ax.grid(True, axis="x", alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "feature_importance.png")
plt.savefig(fig_path, dpi=300)
plt.close()
print("Feature importance figure saved.")

Feature importance figure saved.


In [13]:
# Comparison table vs Yigit 2022

# Contextual comparison with Yigit 2022
print("\nNote: Results are not directly comparable due to differences in datasets and evaluation protocols.")
print(f"\n{'Metric':<20} {'Yigit 2022':<15} {'EquiGuard':<15}")
print("-" * 50)

our_acc       = (y_pred_test == y_test).mean()
our_precision = precision_score(y_test, y_pred_test)
our_recall    = recall_score(y_test, y_pred_test)
our_f1        = f1_score(y_test, y_pred_test)
our_auc       = roc_auc_score(y_test, y_prob_test)

print(f"{'Accuracy':<20} {'94.4%':<15} {our_acc*100:.1f}%")
print(f"{'Precision':<20} {'N/R':<15} {our_precision*100:.1f}%")
print(f"{'Recall':<20} {'N/R':<15} {our_recall*100:.1f}%")
print(f"{'F1 Score':<20} {'N/R':<15} {our_f1*100:.1f}%")
print(f"{'ROC-AUC':<20} {'N/R':<15} {our_auc:.4f}")
print(f"\n{'Dataset':<20} {'16 horses':<15} {'307 horses'}")
print(f"{'Splitting':<20} {'Random':<15} {'Horse-aware'}")
print(f"{'Open source':<20} {'No':<15} {'Yes'}")
print(f"{'Mahalanobis':<20} {'No':<15} {'Yes'}")

# Log final results
log_path = os.path.join(LOGS_DIR, "experiment_log.txt")
with open(log_path, "a") as f:
    f.write("\n=== FINAL TEST RESULTS ===\n")
    f.write(classification_report(y_test, y_pred_test,
            target_names=["Sound", "Lame"]))
    f.write(f"ROC-AUC: {our_auc:.4f}\n")
    f.write(f"Threshold: {best_threshold}\n")

print("\nResults logged.")
print("\nFinal evaluation completed.")


Note: Results are not directly comparable due to differences in datasets and evaluation protocols.

Metric               Yigit 2022      EquiGuard      
--------------------------------------------------
Accuracy             94.4%           92.5%
Precision            N/R             93.1%
Recall               N/R             99.2%
F1 Score             N/R             96.1%
ROC-AUC              N/R             0.8593

Dataset              16 horses       307 horses
Splitting            Random          Horse-aware
Open source          No              Yes
Mahalanobis          No              Yes

Results logged.

Final evaluation completed.
